In [1]:
%pip install langgraph


   -------- ------------------------------- 1/5 [langgraph-sdk]
   ---------------- ----------------------- 2/5 [langgraph-checkpoint]
   ---------------- ----------------------- 2/5 [langgraph-checkpoint]
   ------------------------ --------------- 3/5 [langgraph-prebuilt]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   ---------------------------------------- 5/5 [langgraph]

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_openai import ChatOpenAI

# 모델 초기화
model=ChatOpenAI(model="gpt-4o-mini")
model.invoke('안녕하세요!')

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-CHqAFQdZhoInT2Uggtlub39i1YthR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ab2e41c4-24b9-4557-93b4-f7a04d2c363b-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [3]:
from typing import Annotated    # annotated는 타입 힌트를 사용할 때 사용하는 점수
from typing_extensions import TypedDict # TypedDict는 딕셔너리 타입을 정의할 때 사용하는 함수

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    """
    State 클래스는 TypedDict를 상속받습니다.

    속성:
        messages(Annoted[list[str], add_messages]): 메시지들은 "list" 타입을 가집니다.
        'add_messages' 함수는 이 상태 키가 어떻게 업데이트되어야 하는지를 정의합니다.
    """
    messages: Annotated[list[str], add_messages]

# StateGraph 클래스를 사용하여 State 타입의 그래프 생성
graph_builder=StateGraph(State)

In [4]:
def generate(state:State):
    """
    주어진 상태를 기반으로 챗봇의 응답 메시지를 생성합니다.

    매개변수:
    state (State): 현재 대화 상태를 나타내는 객체로, 이전 메시지들이 포함되어 있습니다.

    반환값:
    dict: 모델이 생성한 응답 메시지를 포함하는 딕셔너리.
        형식은 {"messages": [응답 메시지]}입니다.
    """

    return {"messages": [model.invoke(state["messages"])]}

graph_builder.add_node("generate", generate)

In [5]:
graph_builder.add_edge(START,"generate")
graph_builder.add_edge("generate", END)

graph=graph_builder.compile()

In [6]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass